# Agentic AI for grid operators: a focused Greensboro study
**IEEE SmartGridComm 2026 tutorial companion**  
Based on the supplied proposal by Alaa Selim, Mohamad Chehade and Mahmoud Abdelaal.

This executable lab uses the **SMART-DS Greensboro `uhs20_1247` substation area**, not the full city. Lab 1 isolates its `udt8894` feeder; Labs 2 and 3 retain the three local feeders so the alternate restoration tie has a modeled source. These are synthetic electrical assets, not a verified utility circuit or named neighborhood.

**Learning sequence:** voltage control, restoration, then cyber-physical screening. Follow each proposal through awareness, planning, simulation, verification, refinement and an operator recommendation. No cell dispatches field commands.

The default is a deterministic, executable multi-agent teaching workflow. It does **not** claim to have run an LLM. An optional live Ollama mode uses the same physical evaluator and feedback. Both use fresh OpenDSS solves. No GPU is needed for the default labs.


## 0. Setup and provenance
Unzip the complete bundle, open this notebook in that directory and run all cells. Python 3.10+ is required. Install dependencies in your environment with `python -m pip install -r requirements.txt`. For Colab, upload and unzip the bundle, change into its directory and run the same installation command in a notebook cell. The dataset subset is included.

Source: [OEDI SMART-DS dataset 2981](https://data.openei.org/submissions/2981). Dataset release metadata accompanies the files. Compatibility overlays update retired PV/inverter property names and preserve the original files. The manifest records source hashes.

**Study assumptions:** static snapshots, fixed source voltage and control mode off. Voltage band 0.95–1.05 pu is a chosen tutorial criterion. Line loading uses the model's normal ampere rating. Storage charging is a snapshot action; duration, SOC scheduling and protection coordination require further work.


In [1]:
from pathlib import Path
import os, sys, json, time, hashlib
ROOT = Path.cwd().resolve()
while not (ROOT / 'OPRO_Optimizer.py').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / 'OPRO_Optimizer.py').exists(), 'Open the notebook inside the unzipped bundle.'
sys.path[:0] = [str(ROOT), str(ROOT / 'scripts')]
os.environ['GRIDGUARD_HEADLESS'] = '1'
import pandas as pd
import OPRO_Optimizer as grid
from build_smartds_feeder_compat import build_overlay
import run_tii_restoration_benchmark as restore
import run_tii_fault_injection as cyber
if 'display' not in globals():
    from IPython.display import display
RESULTS = ROOT / 'results'
RESULTS.mkdir(exist_ok=True)
grid.RUN_DIR = RESULTS / 'raw'
grid.RUN_DIR.mkdir(exist_ok=True)
DATA = ROOT / 'SMARTDS_GSO_solar_extreme_batt_high_peak_opendss'
AREA = DATA / 'uhs20_1247'
LEAF = AREA / 'uhs20_1247--udt8894'
VMIN, VMAX = 0.95, 1.05
leaf_manifest = build_overlay(LEAF / 'Master.dss', ROOT / 'generated/voltage')
area_manifest = build_overlay(AREA / 'Master.dss', ROOT / 'generated/restoration')
leaf_master = Path(leaf_manifest['compatibility_master'])
area_master = Path(area_manifest['compatibility_master'])
print('Study area:', AREA.name, ' | voltage feeder:', LEAF.name)
print('Original data modified:', leaf_manifest['original_dataset_modified'])


Study area: uhs20_1247  | voltage feeder: uhs20_1247--udt8894
Original data modified: False


## 1. Agent contracts and the operator boundary
| Agent | Receives | Produces | Benefit demonstrated here |
|---|---|---|---|
| Awareness | Solver snapshot, topology, catalog | State summary and allowed targets | Avoids guessing devices or the operating state |
| Planning | State, objective, action bounds | Structured candidate | Makes each recommendation reproducible |
| Simulation | Candidate and reset model | Voltages, line loading, service | Reveals side effects before recommendation |
| Verification | Candidate, telemetry and simulation | Pass/fail with reasons | Blocks infeasible or untrusted proposals |
| Optimizer | Scored history and rejection reasons | Revised candidate | Repairs failure and reduces unnecessary curtailment |
| Operator | Ranked action with evidence | Approve, reject or request revision | Preserves operational authority |

The voltage lab exposes each role as a function. Restoration and cyber labs use the existing LangGraph engine, whose `initialize`, `plan`, `verify`, `repair`, `accept`, `finalize` trace is displayed. Simulation and verification share an implementation node in that engine; conceptual agents need not be separate processes or separate LLMs. `accept` means acceptance into simulated state, not field authorization.


In [2]:
TRACE = []
def log(agent, event, evidence):
    TRACE.append({'step': len(TRACE)+1, 'agent': agent, 'event': event, 'evidence': evidence})

def solve(case, master, commands):
    return grid.compile_and_solve('tutorial', str(case), str(master), commands, VMIN, VMAX,
                                  recover_on_failure=False)

def awareness_agent(snapshot):
    m = grid.summarize_metrics(snapshot, VMIN, VMAX)
    state = {'metrics': m, 'pv_names': list(snapshot.devices['pvsystems']),
             'topology_hash': snapshot.topology_hash}
    log('Awareness', 'summarize', {'violating_buses': m['viol_buses'], 'pv_count': len(state['pv_names'])})
    return state

def planning_agent(pct, source='deterministic'):
    candidate = {'type': 'pv_fleet_cap', 'pct_pmpp': pct, 'source': source}
    log('Planning', 'propose', candidate.copy())
    return candidate

def simulation_agent(candidate, state):
    # Strict action schema: no raw OpenDSS text from an LLM is accepted.
    if set(candidate) != {'type','pct_pmpp','source'} or candidate['type'] != 'pv_fleet_cap':
        raise ValueError('Invalid action schema')
    pct = candidate['pct_pmpp']
    if isinstance(pct, bool) or not isinstance(pct, (int,float)) or not 0 <= pct <= 100:
        raise ValueError('pct_pmpp must be finite and between 0 and 100')
    commands = ['set controlmode=off'] + [f'Edit PVSystem.{n} %Pmpp={pct}' for n in state['pv_names']]
    snap = solve(LEAF, leaf_master, commands)
    metrics = grid.summarize_metrics(snap, VMIN, VMAX)
    log('Simulation', 'OpenDSS solve', {'pct_pmpp': pct, 'violating_buses': metrics['viol_buses'],
                                     'overloaded_lines': metrics['over_lines']})
    return snap, metrics

def verification_agent(snapshot, metrics, state):
    failures = []
    if not metrics['converged']: failures.append('power flow did not converge')
    if metrics['viol_buses']: failures.append('voltage outside tutorial band')
    if metrics['over_lines']: failures.append('line normal rating exceeded')
    if metrics['served_load_kw'] + .1 < state['metrics']['served_load_kw']: failures.append('service regression')
    if snapshot.topology_hash != state['topology_hash']: failures.append('unexpected topology change')
    log('Verification', 'screen', failures or ['pass modeled snapshot constraints'])
    return not failures, failures


## 2. Lab 1 — voltage control on `udt8894`
The supplied extreme-solar snapshot has overvoltage. The bounded action changes a common real-power cap for the 930 PV systems on this feeder. This is a transparent first exercise in voltage control, not an optimal Volt/VAR controller. The cap is percent of each inverter's `Pmpp`, **not** measured curtailed energy. Inverter kVA is unchanged and reactive-power settings are not edited.

We first try 80%, 60% and 40% caps. Then verifier feedback triggers a 20% probe and midpoint refinement toward a higher feasible cap. More retained PV output is preferred only after the modeled voltage and thermal checks pass. Every probe costs one fresh solve. Feasibility need not be monotonic on other feeders, so this bounded search has no global-optimality guarantee.


In [3]:
baseline = solve(LEAF, leaf_master, ['set controlmode=off'])
state = awareness_agent(baseline)
display(pd.DataFrame([state['metrics']])[['converged','viol_buses','v_severity','over_lines','served_load_kw']])
display(pd.DataFrame([{'device_family': k, 'count':len(v)} for k,v in baseline.devices.items()]))
voltage_rows = []
def evaluate_cap(pct, stage):
    c = planning_agent(pct)
    snap, metrics = simulation_agent(c, state)
    feasible, failures = verification_agent(snap, metrics, state)
    row = {'stage':stage,'pct_pmpp':pct,'feasible':feasible,
           'viol_buses':metrics['viol_buses'],'v_severity':metrics['v_severity'],
           'over_lines':metrics['over_lines'],'thermal_severity':metrics['thermal_severity_pct_sum'],
           'served_load_kw':metrics['served_load_kw'],'failure_reasons':'; '.join(failures)}
    voltage_rows.append(row)
    return row
for pct in [80,60,40]:
    evaluate_cap(pct, 'initial planning')
display(pd.DataFrame(voltage_rows))


,converged,viol_buses,v_severity,over_lines,served_load_kw
0,True,23,0.376088,10,4747.46756


,device_family,count
0,regcontrols,0
1,capacitors,1
2,pvsystems,930
3,invcontrols,547
4,capcontrols,1
5,storages,436
6,storagecontrollers,0
7,generators,0
8,loads,2173
9,lines,1977


,stage,pct_pmpp,feasible,viol_buses,v_severity,over_lines,thermal_severity,served_load_kw,failure_reasons
0,initial planning,80,False,21,0.281688,10,1009.732414,4747.46756,voltage outside tutorial band; line normal rat...
1,initial planning,60,False,7,0.083708,9,503.704579,4747.46756,voltage outside tutorial band; line normal rat...
2,initial planning,40,False,0,0.000000,6,100.521190,4747.46756,line normal rating exceeded


In [4]:
def optimizer_agent(low=20.0, high=40.0, budget=5):
    log('Optimizer', 'repair request', 'Initial candidates fail. Probe lower cap, then retain more PV.')
    low_row = evaluate_cap(low, 'repair probe')
    if not low_row['feasible']:
        return None  # Fail closed; no recommendation from an unverified candidate.
    best = low_row
    for _ in range(budget):
        pct = (low + high) / 2
        row = evaluate_cap(pct, 'refinement')
        if row['feasible']:
            low, best = pct, row
        else:
            high = pct
        log('Optimizer', 'update search interval', {'feasible_lower':low,'upper_probe':high})
    return best
best_voltage = optimizer_agent()
assert best_voltage is not None
display(pd.DataFrame(voltage_rows))
print('Operator recommendation: common PV cap',best_voltage['pct_pmpp'],'% Pmpp')
print('Physical PV setpoint writes:',len(state['pv_names']))
display(pd.DataFrame(TRACE))


,stage,pct_pmpp,feasible,viol_buses,v_severity,over_lines,thermal_severity,served_load_kw,failure_reasons
0,initial planning,80.000,False,21,0.281688,10,1009.732414,4747.46756,voltage outside tutorial band; line normal rat...
1,initial planning,60.000,False,7,0.083708,9,503.704579,4747.46756,voltage outside tutorial band; line normal rat...
2,initial planning,40.000,False,0,0.000000,6,100.521190,4747.46756,line normal rating exceeded
3,repair probe,20.000,True,0,0.000000,0,0.000000,4747.46756,
4,refinement,30.000,True,0,0.000000,0,0.000000,4747.46756,
5,refinement,35.000,False,0,0.000000,3,11.886978,4747.46756,line normal rating exceeded
6,refinement,32.500,True,0,0.000000,0,0.000000,4747.46756,
7,refinement,33.750,False,0,0.000000,2,0.572828,4747.46756,line normal rating exceeded
8,refinement,33.125,True,0,0.000000,0,0.000000,4747.46756,


,step,agent,event,evidence
0,1,Awareness,summarize,"{'violating_buses': 23, 'pv_count': 930}"
1,2,Planning,propose,"{'type': 'pv_fleet_cap', 'pct_pmpp': 80, 'sour..."
2,3,Simulation,OpenDSS solve,"{'pct_pmpp': 80, 'violating_buses': 21, 'overl..."
3,4,Verification,screen,"[voltage outside tutorial band, line normal ra..."
4,5,Planning,propose,"{'type': 'pv_fleet_cap', 'pct_pmpp': 60, 'sour..."
5,6,Simulation,OpenDSS solve,"{'pct_pmpp': 60, 'violating_buses': 7, 'overlo..."
6,7,Verification,screen,"[voltage outside tutorial band, line normal ra..."
7,8,Planning,propose,"{'type': 'pv_fleet_cap', 'pct_pmpp': 40, 'sour..."
8,9,Simulation,OpenDSS solve,"{'pct_pmpp': 40, 'violating_buses': 0, 'overlo..."
9,10,Verification,screen,[line normal rating exceeded]


Operator recommendation: common PV cap 33.125 % Pmpp
Physical PV setpoint writes: 930


### What each additional step buys
The comparison below holds the operating point and action family fixed. The unverified planner's first proposal still has violations. Verification blocks all three initial proposals and returns no action. Refinement finds a passing snapshot and seeks a higher cap. This demonstrates the benefit of the safeguards and search for this case; it does not establish an LLM advantage, operator time savings, or superiority over OPF.


In [5]:
ablation = pd.DataFrame([
 {'workflow':'Planner only','recommended_cap':80,'viol_buses':voltage_rows[0]['viol_buses'],
  'over_lines':voltage_rows[0]['over_lines'],'candidate_solves':1},
 {'workflow':'Planner + verifier','recommended_cap':None,'viol_buses':state['metrics']['viol_buses'],
  'over_lines':state['metrics']['over_lines'],'candidate_solves':3},
 {'workflow':'Planner + verifier + refinement','recommended_cap':best_voltage['pct_pmpp'],
  'viol_buses':best_voltage['viol_buses'],'over_lines':best_voltage['over_lines'],'candidate_solves':len(voltage_rows)}])
display(ablation)
print('Verifier-only retains baseline because no initial candidate passes. Baseline solves excluded.')
pd.DataFrame(voltage_rows).to_csv(RESULTS/'voltage_candidates.csv',index=False)
pd.DataFrame(TRACE).to_json(RESULTS/'voltage_agent_trace.json',orient='records',indent=2)


,workflow,recommended_cap,viol_buses,over_lines,candidate_solves
0,Planner only,80.000,21,10,1
1,Planner + verifier,NaN,23,10,3
2,Planner + verifier + refinement,33.125,0,0,9


Verifier-only retains baseline because no initial candidate passes. Baseline solves excluded.


## 3. Lab 2 — restoration within the same substation area
Retain `udt5173`, `udt8894` and `udt9897` in the parent model. Opening `l(r:udt8894-uhs20_1247)_disconnect` models the outage. The fault remains quarantined. The allowlisted alternate tie is `elbswitch(r:udt8894-udt9916)u_82773`.

Compare the outage, tie closure alone, and coordinated tie plus storage charging. Charging may help in this solar-rich snapshot by absorbing local surplus; always inspect the solver result rather than assuming discharge is beneficial. The existing repair agent adjusts a fleet percentage from measured watch-line loading.

**Acceptance distinction:** the parent network already has violations and nonzero topology diagnostics. Its existing verifier checks improvement relative to the pre-contingency security envelope. It does not prove absolute thermal feasibility or radiality. We show those remaining violations explicitly. This is a static pickup study, not validation of every switching transient or protection setting. No named hospitals or critical facilities are inferred from synthetic load IDs.


In [6]:
fleet = restore.storage_names(restore.STORAGE_FILE)
pre = solve(AREA, area_master, ['set controlmode=off'])
outage = solve(AREA, area_master, ['set controlmode=off',f'Open Line.{restore.FAULT_LINE}'])
pre_m = cyber.compact_metrics(pre)
outage_m = cyber.compact_metrics(outage)
restoration_rows = []
for label, commands in [
 ('Pre-contingency',['set controlmode=off']),
 ('Outage',['set controlmode=off',f'Open Line.{restore.FAULT_LINE}']),
 ('Tie only',['set controlmode=off',f'Open Line.{restore.FAULT_LINE}',f'Edit Line.{restore.TIE_LINE} enabled=yes']),
 ('Tie + 41% charge',['set controlmode=off',f'Open Line.{restore.FAULT_LINE}',f'Edit Line.{restore.TIE_LINE} enabled=yes']+
   [f'Edit Storage.{name} DispMode=EXTERNAL State=CHARGING %Charge=41' for name in fleet])]:
    s = solve(AREA,area_master,commands)
    m = cyber.compact_metrics(s)
    restoration_rows.append({'state':label,'served_load_kw':m['served_load_kw'],
        'restored_from_outage_kw':m['served_load_kw']-outage_m['served_load_kw'],
        'watch_line_pct':restore.line_loading(s,restore.CRITICAL_RECLOSER),
        'viol_buses':m['viol_buses'],'over_lines':m['over_lines']})
display(pd.DataFrame(restoration_rows))


,state,served_load_kw,restored_from_outage_kw,watch_line_pct,viol_buses,over_lines
0,Pre-contingency,18932.31598,4747.46756,48.582426,22,61
1,Outage,14184.84842,0.00000,48.558406,19,50
2,Tie only,18932.31598,4747.46756,138.831484,22,62
3,Tie + 41% charge,18932.31598,4747.46756,99.269267,22,60


In [7]:
# Actual LangGraph run: deterministic physics proposals, simulation, verification and repair.
restoration_result = restore.run_method('physics_seed',101,area_master,RESULTS/'restoration',fleet,
                                      pre_m,pre,outage_m,3,5,'not-used-in-deterministic-mode')
display(pd.DataFrame([restoration_result])[[
 'method','restored_load_kw','restored_load_pct','accepted_fleet_pct_kw',
 'simulated_candidates','final_violating_buses','final_overloaded_lines']])
# Inspect the latest run record and its actual agent handoffs.
run_files = sorted((RESULTS/'restoration/raw').glob('*.json'), key=lambda p:p.stat().st_mtime)
restoration_record = json.loads(run_files[-1].read_text())
display(pd.DataFrame(restoration_record.get('agent_trace', [])))
ledger = restoration_record.get('candidate_evaluation_ledger', [])
display(pd.DataFrame([{k:e.get(k) for k in ['name','certified','simulated','failure_modes','repair_derivation']} for e in ledger]))


,method,restored_load_kw,restored_load_pct,accepted_fleet_pct_kw,simulated_candidates,final_violating_buses,final_overloaded_lines
0,physics_seed,4747.46756,100.0,40.8,15,18,57


,iteration,node,role,actor_type,outputs,route,timestamp_utc,model,prompt_sha256,input_memory_items,output_candidates,historical_duplicates_filtered,physics_fallback_used,physics_portfolio_supplemented,restoration_portfolio_supplemented,adaptive_restoration_repairs,latency_ms,input_candidates,simulated_candidates,certified_candidates,rejected_candidates,invalid_before_simulation,failure_counts,derived_repair_candidates,derivation_methods,candidate,certificate
0,-1,initialize,State and Telemetry Agent,deterministic,"{'violating_buses': 19, 'objective': 477075917...",plan,2026-09-19T18:55:14.144203+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0,plan,OPRO Planner Agent,deterministic,NaN,verify,2026-09-19T18:55:14.197755+00:00,deterministic-no-llm,9b2ac8d5d68bbfa4a125f0ec540c289841ba0ed9afe95c...,1.0,5.0,0.0,True,False,True,0.0,52.8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0,verify,Physics and Certificate Team,deterministic,NaN,repair,2026-09-19T18:55:17.439492+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3241.6,5.0,5.0,0.0,5.0,0.0,NaN,NaN,NaN,NaN,NaN
3,0,repair,Failure Taxonomy Agent,deterministic,NaN,plan,2026-09-19T18:55:17.440032+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'restoration': 3, 'security': 3, 'topology': ...",1.0,[secant_watch_line_repair],NaN,NaN
4,1,plan,OPRO Planner Agent,deterministic,NaN,verify,2026-09-19T18:55:17.473922+00:00,deterministic-no-llm,e871c6b0cdc6ebee0ed7ee9075dd78a351eced3373f00f...,11.0,5.0,5.0,True,False,True,1.0,33.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,1,verify,Physics and Certificate Team,deterministic,NaN,accept,2026-09-19T18:55:20.758303+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3284.0,5.0,5.0,1.0,4.0,0.0,NaN,NaN,NaN,NaN,NaN
6,1,accept,Certificate Commit Agent,deterministic,NaN,plan,2026-09-19T18:55:20.761315+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,RESTORE_REPAIR_secant_charge40p8pct_3e975bdc,"{'device_admissible': True, 'cyber_consistent'..."
7,2,plan,OPRO Planner Agent,deterministic,NaN,verify,2026-09-19T18:55:20.794747+00:00,deterministic-no-llm,84953ef84853027283058125ae9cc62fa09e2024b0eac3...,21.0,5.0,0.0,True,False,True,0.0,33.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2,verify,Physics and Certificate Team,deterministic,NaN,accept,2026-09-19T18:55:24.172830+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3377.7,5.0,5.0,2.0,3.0,0.0,NaN,NaN,NaN,NaN,NaN
9,2,accept,Certificate Commit Agent,deterministic,NaN,finalize,2026-09-19T18:55:24.176231+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,PHY_pv_exact_all_matching_aggressive_2,"{'device_admissible': True, 'cyber_consistent'..."


,name,certified,simulated,failure_modes,repair_derivation
0,RESTORE_blind_elbswitch(r:udt8894-udt9916)u_82773,False,True,[security:thermal_regression],None
1,RESTORE_PROBE_elbswitch(r:udt8894-udt9916)u_82...,False,True,[security:thermal_regression],None
2,PHY_cap_uc92_on,False,True,"[topology:islanding_or_loop_regression, restor...",None
3,PHY_pv_exact_all_matching_aggressive_2,False,True,"[topology:islanding_or_loop_regression, restor...",None
4,PHY_storage_storage_p1ulv16384_charging_50,False,True,"[topology:islanding_or_loop_regression, restor...",None
5,RESTORE_REPAIR_secant_charge40p8pct_3e975bdc,True,True,[],"{'method': 'secant_watch_line_repair', 'operat..."
6,PHY_cap_uc92_off,False,True,"[topology:islanding_or_loop_regression, restor...",None
7,PHY_pv_portfolio_balanced_2,False,True,"[topology:islanding_or_loop_regression, restor...",None
8,PHY_storage_storage_p1ulv16384_charging_100,False,True,"[topology:islanding_or_loop_regression, restor...",None
9,PHY_tap_sb2_uhs20_1247_reg_51_-2,False,True,"[topology:islanding_or_loop_regression, restor...",None


[physics_seed/s101] [GridGuard] Baseline score=4.771e+08 metrics={'converged': True, 'viol_buses': 19, 'viol_type': 'overvoltage', 'deenergized_buses': 2240, 'bus_measurement_count': 5526, 'bus_voltage_coverage_rate': 1.0, 'v_severity': 0.6683376993821692, 'losses_kw': 437.23676576070886, 'max_line_pct': 821.2123063820984, 'line_loading_p95_pct': 20.482818472831383, 'line_rating_count': 4742, 'line_loading_coverage_rate': 1.0, 'over_lines': 50, 'thermal_severity_pct_sum': 7593.8685180449465, 'served_load_kw': 14184.848419999978, 'unserved_load_kw': 4747.467560000007, 'load_multiplier': 1.0, 'isolated_branches': 2249, 'isolated_loads': 3539, 'topology_loops': 208, 'topology_hash': 'f3286592536757d2', 'network_nodes': 5526, 'network_active_edges': 6181, 'network_components': 6, 'network_cycle_rank': 661, 'source_bus_count': 1, 'source_disconnected_loads': 2173, 'recovery_applied': False, 'switching_effort': 0, 'cumulative_device_action_count': 0.0, 'pv_curtailment_command_fraction_sum': 

### Restoration benefit and critical-load extension
Simulation detects the new watch-line overload from blind pickup. Verification rejects a regressive candidate. Repair adds coordinated storage and uses a secant-style update toward 99.5% watch-line loading. The audit trail exposes the reasoning and the accepted physical action count.

The supplied case has no validated critical-load labels. For a research extension, supply an operator-reviewed load-to-priority map and maximize weighted restored load, subject to the same checks. Do not interpret this notebook's equal-weight nominal restored kW as critical-load optimization. Battery energy sufficiency over a restoration horizon is also outside this static demonstration.


## 4. Lab 3 — controlled cyber-physical injections
Run a known feasible control and ten negative controls through the real verifier. Cases include stale telemetry, a revoked trust flag, topology mismatch, quarantined targets, unknown actions/devices, actuator readback failures and a physically regressive tie proposal.

These are **software-level fault injections**, not attacks on a live system. A supplied `trusted` flag is a policy input, not an implemented authentication system. Topology fingerprints detect disagreement with the expected state, not an attacker who compromises both copies. Arbitrary false-data injection, stealth attacks and prompt-injection robustness need separate evaluation. The readback cases deliberately mock the interface; the other simulated candidates receive fresh OpenDSS solves.


In [8]:
config = cyber.build_config(fleet)
grid.RUN_DIR = RESULTS/'cyber/raw'
grid.RUN_DIR.mkdir(parents=True,exist_ok=True)
cyber_rows = []
for scenario in cyber.scenarios():
    row = cyber.evaluate_scenario(config,scenario,outage,outage_m,pre,pre_m,area_master,
                                  grid.compile_and_solve)
    cyber_rows.append(row)
display(pd.DataFrame(cyber_rows)[['scenario_id','expected_disposition','observed_disposition',
                                 'simulated','unsafe_commit','failure_modes']])
cyber_summary = cyber.summarize(cyber_rows)
print(json.dumps(cyber_summary,indent=2))
assert all(r['detected'] for r in cyber_rows), 'An injection did not produce its expected disposition.'
assert not any(r['unsafe_commit'] for r in cyber_rows)


,scenario_id,expected_disposition,observed_disposition,simulated,unsafe_commit,failure_modes
0,known_safe_control,certify,certify,True,False,[]
1,unknown_action_schema,reject,reject,False,False,[action[0]:unknown_type:raw_dss]
2,unknown_device,reject,reject,False,False,[action[0]:unknown_der:PVSystem.NOT_IN_CATALOG]
3,unsafe_vsource_attempt,reject,reject,False,False,[action[0]:unknown_der:Vsource.source]
4,stale_telemetry,reject,reject,False,False,[cyber:stale_telemetry:843159327s]
5,untrusted_telemetry,reject,reject,False,False,[cyber:telemetry_untrusted]
6,quarantined_target,reject,reject,False,False,[action[0]:cyber_quarantined:line.l(r:udt8894-...
7,topology_hash_mismatch,reject,reject,False,False,[cyber:topology_mismatch]
8,actuation_readback_mismatch,reject,reject,True,False,[cyber:actuation_mismatch:action[0]:line.elbsw...
9,actuation_readback_unavailable,reject,reject,True,False,[cyber:actuation_unavailable:action[0]:line.el...


{
  "scenario_count": 11,
  "hazard_case_count": 10,
  "hazards_detected": 10,
  "hazard_detection_rate": 1.0,
  "safe_control_count": 1,
  "false_reject_count": 0,
  "false_reject_rate": 0.0,
  "unsafe_commit_count": 0,
  "pre_simulation_rejection_count": 7,
  "post_simulation_rejection_count": 3,
  "by_layer": [
    {
      "layer": "actuation",
      "cases": 2,
      "detected": 2,
      "detection_rate": 1.0,
      "pre_simulation_rejections": 0
    },
    {
      "layer": "admissibility",
      "cases": 2,
      "detected": 2,
      "detection_rate": 1.0,
      "pre_simulation_rejections": 2
    },
    {
      "layer": "control",
      "cases": 1,
      "detected": 1,
      "detection_rate": 1.0,
      "pre_simulation_rejections": 0
    },
    {
      "layer": "cyber",
      "cases": 4,
      "detected": 4,
      "detection_rate": 1.0,
      "pre_simulation_rejections": 4
    },
    {
      "layer": "physics",
      "cases": 1,
      "detected": 1,
      "detection_rate": 1.0,
  

### Step-by-step interpretation
1. Awareness carries freshness, trust, quarantine and topology evidence with the state.
2. The planner submits the same structured proposal under changed evidence.
3. Early checks reject malformed or untrusted input before paying for a solve.
4. Simulation exposes thermal regression in an otherwise executable tie action.
5. Verification prevents a rejected candidate reaching simulated acceptance.
6. The repair route returns a reason to planning; it never bypasses the verifier.

Report positive-control false rejects separately from negative-control rejections. This small designed suite checks implementation behavior; its pass rate is not a real-world cyberattack detection rate.


## 5. Optional live LLM planning and verifier-guided refinement
Default execution below is off. Start a local Ollama server with an installed Llama or Qwen model, set `OLLAMA_MODEL` to its exact installed identifier, then enable the switch. No model-family performance comparison is claimed by the default output.

The prompt supplies a bounded numeric action schema, operating metrics and prior scores. The model proposes only a PV cap. Python validates the response, performs the physical solve and sends the rejection reasons back. Model text cannot execute arbitrary commands. The optional restoration run uses the existing planner and critic with the same allowlisted actions.


In [9]:
RUN_LIVE_LLM = False
if RUN_LIVE_LLM:
    assert os.environ.get('OLLAMA_MODEL'), 'Set OLLAMA_MODEL to an installed model.'
    llm = grid.LLMClient(seed=101)
    memory = []
    for iteration in range(6):
        response = llm.chat(
          system='You plan PV caps for a simulation. Return only JSON with one key pct_pmpp, a number in [0,100]. Maximize the cap after all voltage and thermal constraints pass.',
          user_text=json.dumps({'baseline':state['metrics'],'scored_history':memory}),
          temperature=0.2,num_predict=256,response_format='json')
        try:
            parsed = json.loads(response)
            if set(parsed) != {'pct_pmpp'}: raise ValueError('Unexpected keys')
            candidate = planning_agent(parsed['pct_pmpp'],source='live LLM')
            snapshot, metrics = simulation_agent(candidate,state)
            passed, reasons = verification_agent(snapshot,metrics,state)
            memory.append({'iteration':iteration,'pct_pmpp':parsed['pct_pmpp'],
                           'passed':passed,'reasons':reasons,'voltage_severity':metrics['v_severity'],
                           'overloaded_lines':metrics['over_lines']})
        except (ValueError,TypeError) as error:
            memory.append({'iteration':iteration,'passed':False,'reasons':[str(error)]})
    display(pd.DataFrame(memory))
    feasible = [r for r in memory if r['passed']]
    print('Best verified live proposal:',max(feasible,key=lambda r:r['pct_pmpp']) if feasible else 'No recommendation')
    (RESULTS/'live_llm_voltage.json').write_text(json.dumps({'model':llm.model,'history':memory},indent=2))
    live_restore = restore.run_method('gridguard_hybrid',101,area_master,RESULTS/'live_restoration',fleet,
        pre_m,pre,outage_m,3,5,llm.model)
    display(pd.DataFrame([live_restore]))
else:
    print('Live LLM calls skipped. All displayed primary results use deterministic policies.')


Live LLM calls skipped. All displayed primary results use deterministic policies.


## 6. Evidence, exercises and research design
**What this run establishes:** reproducible simulations on a real SMART-DS subset, explicit agent handoffs, candidate rejection, bounded repair and audit records. **What remains a hypothesis:** LLMs improve decision quality or reduce operator effort compared with conventional optimization or a single agent.

For a publication study, compare no action, a deterministic search, OPF/Volt-VAR control, an LLM without verification, a verified LLM and a verified LLM with feedback. Match physical action limits and simulation budgets. Separate unsafe proposals from unsafe simulated commitments. Use multiple operating points, outages and model seeds; report uncertainty, wall time, tokens, solver calls, losses, actual PV energy and remaining violations. Do not treat the unequal-budget teaching ablation as a fair optimizer benchmark.

**Exercises**
1. Change the voltage band and rerun the complete voltage section. Does the previous cap still pass?
2. Limit refinement to two probes. Compare the accepted cap and solve count.
3. Compare tie-only and coordinated pickup. Explain why charging helps in this snapshot.
4. Change a negative-control input to a valid one and inspect its trace.
5. Enable live LLM mode for two installed model families with the same six-solve budget and seeds. Separate parse failures from physical failures.

**Three-hour teaching plan:** motivation 20 min, multi-agent workflow 25, open-weight models 20, optimizer patterns 20, voltage lab 25, restoration lab 25, cyber lab 25, discussion 20.


In [10]:
summary = {
 'scope':'SMART-DS Greensboro uhs20_1247; voltage lab uses udt8894 leaf',
 'execution_mode':'deterministic OpenDSS and LangGraph; optional live LLM separately labeled',
 'voltage_baseline':state['metrics'],'voltage_best':best_voltage,
 'voltage_candidates':voltage_rows,'restoration_states':restoration_rows,
 'restoration_result':restoration_result,'cyber_trials':cyber_rows,'cyber_summary':cyber_summary,
 'source_master_sha256':area_manifest['source_master_sha256'],
 'source_tree_sha256':area_manifest['redirected_source_tree_sha256'],
 'gridguard_sha256':hashlib.sha256((ROOT/'OPRO_Optimizer.py').read_bytes()).hexdigest(),
 'limitations':['static snapshots','synthetic network','parent model has pre-existing violations',
               'no proof of absolute radiality','no critical-load labels','no field dispatch',
               'no live LLM performance claim from default execution']}
(RESULTS/'study_summary.json').write_text(json.dumps(summary,indent=2,allow_nan=False))
pd.DataFrame(restoration_rows).to_csv(RESULTS/'restoration_states.csv',index=False)
pd.DataFrame(cyber_rows).to_csv(RESULTS/'cyber_trials.csv',index=False)
print('Saved machine-readable evidence to', RESULTS)


Saved machine-readable evidence to C:\Users\f00863x\OPRO\output\smartgridcomm_greensboro\results


## References and implementation traceability
- User-supplied *Tutorial Proposal for IEEE SmartGridComm 2026*, sections 2–5: roles, workflow, labs and timing.
- [SMART-DS Synthetic Electrical Network Data](https://data.openei.org/submissions/2981), Palmintier et al., OEDI dataset 2981: synthetic Greensboro network provenance.
- Bundled `OPRO_Optimizer.py`: solver wrapper, action schemas, policy checks and LangGraph engine.
- Bundled restoration and controlled-injection runners: original scenario and verification logic. This notebook reruns the scenarios rather than presenting old run files as new measurements.

Retain the source manifests and raw candidate ledgers when adapting the tutorial. Revalidate any new control, line rating, SOC horizon, topology model or critical-load priority before making stronger operational claims.
